# 02 - Exploratory analysis

The purpose of this notebook is to work out what structure exists in the series before
fitting anything, and in particular to form a prior about which benchmark is likely to be
hard to beat.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting

hourly = data.load_hourly()
y = hourly[config.TARGET]
train, test = data.train_test_split(y)

## The series

In [ ]:
fig = plotting.plot_series_overview(hourly)
fig

Two things stand out. The series is spiky rather than smooth: long quiet stretches
around 50 Wh punctuated by short bursts of several hundred. And the bursts are not
regular. There is a clear daily rhythm underneath, but the size and exact timing of the
peaks varies a lot from day to day.

## Distribution and seasonal profiles

In [ ]:
fig = plotting.plot_seasonal_profiles(hourly)
fig

In [ ]:
print("Distribution of hourly appliance energy use")
print(y.describe().round(1))
print(f"\nSkewness: {y.skew():.2f}")
print(f"Share of hours below 60 Wh: {(y < 60).mean():.1%}")
print(f"Share of hours above 200 Wh: {(y > 200).mean():.1%}")

The distribution is heavily right-skewed (skewness 2.4). Nearly half of all hours sit
below 60 Wh, which is effectively the standby load of the house, while a small minority
of hours carry most of the variation.

The hour-of-day profile is the dominant structure: a flat overnight trough from about
01:00 to 05:00, a morning rise, and a broad evening peak around 17:00 to 20:00. The
interquartile band is wide during the day and narrow at night, which already tells us
that daytime hours will be much harder to forecast.

## Day of week

In [ ]:
by_dow = y.groupby(y.index.dayofweek).agg(["mean", "median", "std"])
by_dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_dow.round(1)

Weekend days are somewhat higher on average than weekdays, but the difference is small
relative to the within-day variation. This is a single household, so the weekly pattern
reflects one family's habits rather than any aggregate regularity.

## Autocorrelation

The ACF and PACF tell us what the SARIMAX specification needs to capture.

In [ ]:
fig = plotting.plot_acf_pacf(train, lags=200)
fig

In [ ]:
from statsmodels.tsa.stattools import acf

values = acf(train, nlags=200, fft=True)

for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
    print(f"lag {lag:>4}: {values[lag]:.3f}")

Autocorrelation decays quickly at short lags but there are clear local peaks at lag 24
and lag 168, confirming both daily and weekly seasonality. The lag-24 correlation
(around 0.4) is stronger than the lag-168 correlation, but neither is close to 1, so a
seasonal naive forecast will be a mediocre predictor.

That is an important early signal: the seasonality is real but weak, so we should not
expect any model to achieve a low MASE.

## Stationarity

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

adf_stat, adf_p = adfuller(train, autolag="AIC")[:2]
kpss_stat, kpss_p = kpss(train, regression="c", nlags="auto")[:2]

print(f"ADF statistic {adf_stat:.2f}, p-value {adf_p:.4f}")
print(f"KPSS statistic {kpss_stat:.3f}, p-value {kpss_p:.3f}")

The ADF test rejects a unit root decisively and KPSS does not reject stationarity around
a constant, so the series does not need ordinary differencing. Seasonal differencing at
lag 24 is still worth applying to remove the daily cycle, which is what the SARIMAX
specification does.

## Relationship with the covariates

In [ ]:
fig = plotting.plot_correlations(hourly)
fig

In [ ]:
correlations = hourly.corr(numeric_only=True)[config.TARGET].drop(config.TARGET)
print("Strongest absolute correlations with the target:")
print(correlations.reindex(correlations.abs().sort_values(ascending=False).index).head(10).round(3))

No covariate is strongly related to the target. The `lights` channel is the best of them,
which is unsurprising since lights being on is a proxy for someone being at home, but even
that is a moderate relationship. Outdoor temperature and humidity are weak.

This is the first concrete warning that the weather covariates may not earn their place in
a forecasting model, and it is borne out later: adding them makes both SARIMAX and the
gradient boosting model worse.

## What the exploration implies for modelling

1. The dominant structure is hour of day, so any model that captures the daily profile
   will do most of the work.
2. Seasonality is weak in absolute terms, so MASE values close to 1 should be expected,
   and large improvements over benchmarks are unlikely.
3. The series is spiky and right-skewed. Models that forecast a conditional mean or median
   will systematically undershoot peaks, which will show up as negative bias.
4. Covariates are weakly correlated with the target and should be treated sceptically.